# Mission 2B: Rebuild Mary's Language Core

In [28]:
import math, random
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print('Seed:', seed)

Seed: 42


## Task A — Reflex arc

Derive the backward pass for the supplied two-layer ReLU network, check four gradients with centered differences, then train for 100 gradient-descent steps.

In [29]:
rng_a = np.random.default_rng(seed)
N, d, h, C = 8, 3, 5, 3
X = rng_a.normal(size=(N, d))
y = rng_a.integers(0, C, N)
params = {
    'W1': rng_a.normal(size=(d, h)) * 0.5,
    'b1': np.zeros(h),
    'W2': rng_a.normal(size=(h, C)) * 0.5,
    'b2': np.zeros(C),
}

def forward(params):
    Z1 = X @ params['W1'] + params['b1']
    A1 = np.maximum(0, Z1)
    Z2 = A1 @ params['W2'] + params['b2']
    Z2 = Z2 - Z2.max(axis=1, keepdims=True)
    P = np.exp(Z2)
    P /= P.sum(axis=1, keepdims=True)
    loss = -np.log(P[np.arange(N), y]).mean()
    return loss, (Z1, A1, P)

def backward(params, cache):
    Z1, A1, P = cache
    dZ2 = P.copy()
    dZ2[np.arange(N), y] -= 1
    dZ2 /= N
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ params['W2'].T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

def numeric_grad(params, name, idx, eps=1e-5):
    original = params[name][idx]
    try:
        params[name][idx] = original + eps
        loss_plus, _ = forward(params)
        params[name][idx] = original - eps
        loss_minus, _ = forward(params)
    finally:
        params[name][idx] = original
    return (loss_plus - loss_minus) / (2 * eps)

loss, cache = forward(params)
grads = backward(params, cache)
checks = [('W1', (0, 0)), ('W1', (d - 1, h - 1)), ('W2', (2, 1)), ('b2', (0,))]
relative_errors = []
for name, idx in checks:
    analytic = grads[name][idx]
    numeric = numeric_grad(params, name, idx)
    rel = abs(analytic - numeric) / max(abs(analytic) + abs(numeric), 1e-12)
    relative_errors.append(rel)
    print(name, idx, 'analytic', round(analytic, 6), 'numeric', round(numeric, 6), 'relative error', f'{rel:.2e}')
assert max(relative_errors) < 1e-6, 'Gradient check failed'

print('Loss checkpoints (baseline log(C) =', round(math.log(C), 6), ')')
for step in range(100):
    loss, cache = forward(params)
    if step in (0, 50, 99):
        print(f'step {step}: loss = {loss:.6f}')
    step_grads = backward(params, cache)
    for name in params:
        params[name] -= 0.5 * step_grads[name]

W1 (0, 0) analytic -0.001952 numeric -0.001952 relative error 2.74e-10
W1 (2, 4) analytic -0.028503 numeric -0.028503 relative error 4.20e-11
W2 (2, 1) analytic -0.148531 numeric -0.148531 relative error 2.28e-11
b2 (0,) analytic 0.044814 numeric 0.044814 relative error 1.01e-11
Loss checkpoints (baseline log(C) = 1.098612 )
step 0: loss = 1.048768
step 50: loss = 0.100760
step 99: loss = 0.041384


**Verdict:** All four sampled analytic gradients agree with their centered-difference estimates to relative error below 1e-6, so the backward pass passes the numerical check. The loss checkpoints are compared with the uniform-prediction baseline log(C) in the output above.

## Task B — Rebuild the language core

Generate the supplied 400-line ship log, compare an add-one bigram model with a three-word neural language model, then inspect temperature-scaled predictions and samples.

In [30]:
rng = random.Random(seed)
sections = ['engine bay', 'cargo hold', 'lab', 'bridge', 'airlock', 'reactor room']
findings = ['astrophage', 'a cracked pipe', 'loose cables', 'a warm panel', 'nothing unusual']
templates = [
    'mary scans the {sec} and finds {obj} .',
    'grace checks the {sec} for {obj} .',
    'rocky hums near the {sec} and finds {obj} .',
    'the crew repairs the {sec} after {obj} .',
]
log = [rng.choice(templates).format(sec=rng.choice(sections), obj=rng.choice(findings)) for _ in range(400)]
tokens = ' '.join(log).split()
split = int(len(tokens) * 0.8)
train_tokens, test_tokens = tokens[:split], tokens[split:]
vocab = sorted(set(tokens))
stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for word, i in stoi.items()}
V = len(vocab)
K = 3
print('Vocabulary:', V, '| train tokens:', len(train_tokens), '| test tokens:', len(test_tokens))
print('First 12 tokens:', ' '.join(tokens[:12]))

Vocabulary: 34 | train tokens: 3128 | test tokens: 782
First 12 tokens: mary scans the engine bay and finds loose cables . grace checks


In [31]:
bigram_counts = defaultdict(Counter)
unigram_counts = Counter(train_tokens)
for prev_word, next_word in zip(train_tokens[:-1], train_tokens[1:]):
    bigram_counts[prev_word][next_word] += 1

def bigram_probability(prev_word, next_word):
    return (bigram_counts[prev_word][next_word] + 1) / (unigram_counts[prev_word] + V)

bigram_log_probs = [
    math.log(bigram_probability(test_tokens[i - 1], test_tokens[i]))
    for i in range(K, len(test_tokens))
]
bigram_test_ppl = math.exp(-sum(bigram_log_probs) / len(bigram_log_probs))
print(f'B1 add-one bigram test perplexity (test_tokens[K:]): {bigram_test_ppl:.4f}')

B1 add-one bigram test perplexity (test_tokens[K:]): 2.5311


In [32]:
def make_windows(sequence):
    contexts, targets = [], []
    for i in range(K, len(sequence)):
        contexts.append([stoi[w] for w in sequence[i - K:i]])
        targets.append(stoi[sequence[i]])
    return (torch.tensor(contexts, dtype=torch.long),
            torch.tensor(targets, dtype=torch.long))

X_train, y_train = make_windows(train_tokens)
X_test, y_test = make_windows(test_tokens)  # predicts exactly test_tokens[K:]
assert len(y_test) == len(test_tokens[K:])

class MaryLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 16)
        self.hidden = nn.Linear(K * 16, 64)
        self.output = nn.Linear(64, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x).reshape(x.shape[0], K * 16)
        return self.output(torch.tanh(self.hidden(embedded)))

torch.manual_seed(seed)
model = MaryLM(V)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()
    train_logits = model(X_train)
    train_loss = criterion(train_logits, y_train)
    train_loss.backward()
    optimizer.step()
    if epoch in (50, 100, 150, 200):
        model.eval()
        with torch.no_grad():
            train_ppl = torch.exp(criterion(model(X_train), y_train)).item()
            test_ppl = torch.exp(criterion(model(X_test), y_test)).item()
        print(f'Epoch {epoch:3d} | train perplexity {train_ppl:.4f} | test perplexity {test_ppl:.4f}')

model.eval()
with torch.no_grad():
    neural_test_ppl = torch.exp(criterion(model(X_test), y_test)).item()
print(f'Final neural test perplexity: {neural_test_ppl:.4f}')

Epoch  50 | train perplexity 1.7041 | test perplexity 1.7166
Epoch 100 | train perplexity 1.6960 | test perplexity 1.7215
Epoch 150 | train perplexity 1.6942 | test perplexity 1.7246
Epoch 200 | train perplexity 1.6934 | test perplexity 1.7263
Final neural test perplexity: 1.7263


In [33]:
print(f'{"Model":<20} {"Test perplexity":>16}')
print('-' * 38)
for name, value in [('Add-one bigram', bigram_test_ppl), ('Neural MaryLM', neural_test_ppl)]:
    print(f'{name:<20} {value:16.4f}')

Model                 Test perplexity
--------------------------------------
Add-one bigram                 2.5311
Neural MaryLM                  1.7263


**B3 verdict:** The neural MaryLM wins on this test set with perplexity 1.7263, lower than the add-one bigram model’s 2.5311. Its three-word context and learned embeddings help it represent recurring phrases, while add-one smoothing protects the shorter-context bigram model from zero probabilities but does not match MaryLM here.

In [34]:
def temperature_distribution(context_words, temperature):
    if temperature <= 0:
        raise ValueError('temperature must be positive')
    context_ids = torch.tensor([[stoi[w] for w in context_words]], dtype=torch.long)
    with torch.no_grad():
        scaled_logits = model(context_ids)[0] / temperature
        probs = torch.softmax(scaled_logits, dim=0)
    values, indices = torch.topk(probs, 3)
    return [(itos[i.item()], p.item()) for p, i in zip(values, indices)]

context = ['bay', 'and', 'finds']
for temperature in (1.0, 0.5):
    print(f'Top 3 after {context} at temperature {temperature}:')
    for word, probability in temperature_distribution(context, temperature):
        print(f'  {word:<18} {probability:.6f}')

def generate_more(start_words, temperature, number_of_words=12):
    generated = list(start_words)
    for _ in range(number_of_words):
        context_ids = torch.tensor([[stoi[w] for w in generated[-K:]]], dtype=torch.long)
        with torch.no_grad():
            probs = torch.softmax(model(context_ids)[0] / temperature, dim=0)
        next_id = torch.multinomial(probs, num_samples=1).item()
        generated.append(itos[next_id])
    return generated

for temperature in (0.5, 1.5):
    torch.manual_seed(seed)
    print(f'12-word continuation at temperature {temperature}:',
          ' '.join(generate_more(['mary', 'scans', 'the'], temperature, 12)))

Top 3 after ['bay', 'and', 'finds'] at temperature 1.0:
  a                  0.532650
  nothing            0.233707
  astrophage         0.200700
Top 3 after ['bay', 'and', 'finds'] at temperature 0.5:
  a                  0.747228
  nothing            0.143851
  astrophage         0.106087
12-word continuation at temperature 0.5: mary scans the airlock and finds a warm panel . mary scans the reactor room
12-word continuation at temperature 1.5: mary scans the airlock and finds astrophage . the crew repairs the lab after a


**B4(c):** Lowering temperature from 1.0 to 0.5 sharpens the distribution, increasing the probability of the most likely next words and reducing probability assigned to alternatives. At 1.5 the distribution is flatter, so the sampled continuation is generally more varied than at 0.5.

## Task C — The emergency filter

Compare both distress detectors with class-specific and macro-averaged scores, then compute inverse-frequency class weights.

In [35]:
model_a = dict(TP=10, FN=40, FP=5, TN=945)
model_b = dict(TP=40, FN=10, FP=60, TN=890)

def report(m):
    TP, FN, FP, TN = m['TP'], m['FN'], m['FP'], m['TN']
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0
    distress_f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    routine_f1 = 2 * TN / (2 * TN + FN + FP) if 2 * TN + FN + FP else 0.0
    macro_f1 = (distress_f1 + routine_f1) / 2
    return {'accuracy': accuracy, 'distress_precision': precision,
            'distress_recall': recall, 'distress_f1': distress_f1,
            'routine_f1': routine_f1, 'macro_f1': macro_f1}

for name, counts in [('Model A', model_a), ('Model B', model_b)]:
    print(name)
    for metric, value in report(counts).items():
        print(f'  {metric}: {value:.4f}')

class_counts = [950, 50]
total = sum(class_counts)
class_weights = torch.tensor(
    [total / (len(class_counts) * count) for count in class_counts],
    dtype=torch.float32
)
print('Class weights for [routine, distress]:', class_weights.tolist())
weighted_loss = nn.CrossEntropyLoss(weight=class_weights)
print('Loss function:', weighted_loss)

Model A
  accuracy: 0.9550
  distress_precision: 0.6667
  distress_recall: 0.2000
  distress_f1: 0.3077
  routine_f1: 0.9767
  macro_f1: 0.6422
Model B
  accuracy: 0.9300
  distress_precision: 0.4000
  distress_recall: 0.8000
  distress_f1: 0.5333
  routine_f1: 0.9622
  macro_f1: 0.7477
Class weights for [routine, distress]: [0.5263158082962036, 10.0]
Loss function: CrossEntropyLoss()


**C(c) verdict:** Mary should deploy Model B: it catches 40 of 50 distress calls (recall 0.80) and has macro-F1 0.7478, versus 10 of 50 (recall 0.20) and macro-F1 0.6422 for Model A. Model A’s accuracy is higher (0.955 versus 0.930) only because routine calls dominate the data; that number hides its 40 missed distress calls, compared with 10 for Model B.

## Task D — The black-box recorder

Inspect the 12-epoch training log, identify the lowest validation loss, measure the final generalization gap, and apply patience=2 early stopping.

In [36]:
best = 6  # hidden from the analyst in the assignment
recorder_epochs = np.arange(1, 13)
train_loss = 0.7 * np.exp(-0.28 * recorder_epochs)
val_loss = (0.7 * np.exp(-0.28 * np.minimum(recorder_epochs, best))
            + 0.03 + 0.02 * np.maximum(0, recorder_epochs - best) ** 1.5)

print(f'{"epoch":>5} {"train loss":>12} {"validation loss":>17}')
for epoch, tr, va in zip(recorder_epochs, train_loss, val_loss):
    print(f'{epoch:5d} {tr:12.3f} {va:17.3f}')

best_val_epoch = int(recorder_epochs[np.argmin(val_loss)])
gap_at_12 = val_loss[-1] - train_loss[-1]
print('Lowest-validation-loss epoch:', best_val_epoch)
print(f'Validation-minus-train gap at epoch 12: {gap_at_12:.6f}')

patience = 2
running_best = float('inf')
restored_epoch = None
wait = 0
stopped_epoch = None
for epoch, current_val in zip(recorder_epochs, val_loss):
    if current_val < running_best:
        running_best = current_val
        restored_epoch = int(epoch)
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            stopped_epoch = int(epoch)
            break
print(f'With patience={patience}, stop at epoch {stopped_epoch}; restore epoch {restored_epoch} weights.')

epoch   train loss   validation loss
    1        0.529             0.559
    2        0.400             0.430
    3        0.302             0.332
    4        0.228             0.258
    5        0.173             0.203
    6        0.130             0.160
    7        0.099             0.180
    8        0.075             0.217
    9        0.056             0.264
   10        0.043             0.320
   11        0.032             0.384
   12        0.024             0.454
Lowest-validation-loss epoch: 6
Validation-minus-train gap at epoch 12: 0.430086
With patience=2, stop at epoch 8; restore epoch 6 weights.


**D(d–e) diagnosis:** Validation loss reaches its minimum at epoch 6 and then rises while training loss keeps falling, which indicates overfitting; dropout and weight decay are two ways to reduce it. With patience=2, the second consecutive validation non-improvement occurs at epoch 8, so training stops there and restores the best weights from epoch 6.